# Thinking vs no thinking: which answers are right?

The same 20 recipes went through Claude Haiku 5.5 twice: once with its default thinking, once with thinking turned off. Everything else is identical. **The decision:** can we run the 212,846 recipes without thinking (about 40% cheaper)?

**How to review (about 15 minutes):**
1. Run all cells once. No API calls are made.
2. For each recipe below, read the steps and the ingredient table. Rows with a yellow background and `≠` are where the two modes disagree.
3. In the small cell under the recipe, set the verdict: `thinking`, `nothinking`, `both` (equally fine) or `neither`. A note is optional.
4. Run the last cell. It saves your 20 verdicts to `review.csv` and prints the tally.

Many recipes say "mix all ingredients" with no amounts, so sometimes nothing settles it. Pick `both` if the differences don't matter, or write "can't tell" in the note.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd()
assert (HERE / "stage1_rows.json").exists(), "open this notebook from pipelines/foodcom/pilot/"

stage1 = {r["source_ref"]: r for r in json.loads((HERE / "stage1_rows.json").read_text())}
order = list(stage1)  # equal recipes first, then more, then fewer


def load(mode):
    lines = (HERE / f"results_{mode}.jsonl").read_text().splitlines()
    return {r["source_ref"]: r for r in map(json.loads, lines)}


runs = {"thinking": load("thinking"), "nothinking": load("nothinking")}
PRICE_IN, PRICE_OUT = 0.10, 0.50  # USD per million tokens, same as augment.py
SHARE = {"equal": 0.211, "more": 0.747, "fewer": 0.042}  # real mix of the 212,846 recipes
FULL_RUN = 212_846
verdicts, notes = {}, {}


def cost(rec):
    return (rec["input_tokens"] * PRICE_IN + rec["output_tokens"] * PRICE_OUT) / 1e6


def fmt(e):
    """'2 tbsp', or just 'to_taste' when there is no amount."""
    return e["unit"] if e["quantity"] is None else f"{e['quantity']:g} {e['unit']}"


def same(x, y):
    """Same unit and quantity within 1%, so 0.33 and 0.3333 cup count as equal."""
    if x["unit"] != y["unit"]:
        return False
    if x["quantity"] is None or y["quantity"] is None:
        return x["quantity"] == y["quantity"]
    return abs(x["quantity"] - y["quantity"]) <= 0.01 * max(x["quantity"], y["quantity"])


def pair_extras(a, b):
    """Pair extras whose names contain one another (oil / cooking oil) across the modes."""
    left, pairs = list(b), []
    for x in a:
        n = x["name"].lower()
        hit = next((y for y in left if n in y["name"].lower() or y["name"].lower() in n), None)
        if hit:
            left.remove(hit)
        pairs.append((x["name"] if not hit or hit["name"].lower() == n else f"{x['name']} / {hit['name']}", x, hit))
    return pairs + [(y["name"], None, y) for y in left]


def answers(key):
    return runs["thinking"][key]["result"], runs["nothinking"][key]["result"]


def n_differ(key):
    a, b = answers(key)
    return sum(not same(x, y) for x, y in zip(a["ingredients"], b["ingredients"]))


print(len(order), "recipes loaded")

20 recipes loaded


## At a glance

In [2]:
rows = []
for mode, recs in runs.items():
    rs = list(recs.values())
    by_group = {}
    for r in rs:
        by_group.setdefault(stage1[r["source_ref"]]["group"], []).append(cost(r))
    per = sum(SHARE[g] * sum(c) / len(c) for g, c in by_group.items())
    rows.append({
        "mode": mode,
        "valid": f"{sum(r['status'] == 'ok' for r in rs)}/20",
        "retries": sum(r["attempts"] > 1 for r in rs),
        "output tokens / recipe": round(sum(r["output_tokens"] for r in rs) / 20),
        "full run, direct": f"${per * FULL_RUN:,.0f}",
        "full run, Batches": f"${per * FULL_RUN / 2:,.0f}",
    })
display(pd.DataFrame(rows).set_index("mode"))

total = sum(len(answers(k)[0]["ingredients"]) for k in order)
diff = sum(n_differ(k) for k in order)
cuisine_diff = sum(answers(k)[0]["cuisines"] != answers(k)[1]["cuisines"] for k in order)
display(Markdown(
    f"The modes give a different amount or unit on **{diff} of {total}** ingredient entries ({diff / total:.0%}) "
    f"and a different cuisine set on {cuisine_diff} of 20 recipes. For scale: two *identical* thinking runs differed on "
    "about a third of entries in an earlier 5-recipe test, so a difference alone does not mean one mode is worse. "
    "Costs are reweighted to the real mix of recipe types and are rough (20 recipes)."))

,valid,retries,output tokens / recipe,"full run, direct","full run, Batches"
mode,,,,,
thinking,20/20,0,932,$160,$80
nothinking,20/20,2,288,$93,$46


The modes give a different amount or unit on **49 of 170** ingredient entries (29%) and a different cuisine set on 9 of 20 recipes. For scale: two *identical* thinking runs differed on about a third of entries in an earlier 5-recipe test, so a difference alone does not mean one mode is worse. Costs are reweighted to the real mix of recipe types and are rough (20 recipes).

## Review the 20 recipes

In [3]:
def card(i, key):
    s = stage1[key]
    a, b = answers(key)
    names, quantities = s["ingredient_parts"], s["quantities_raw"]
    steps = "\n".join(f"{n}. {t}" for n, t in enumerate(s["instructions"], 1))
    note = "" if s["group"] == "equal" else "  \n_Raw quantities may be shifted for this recipe, so lean on the steps._"
    display(Markdown(f"### {i}. {s['name']}  \n{s['group']} · {len(names)} ingredients, {len(quantities)} raw quantities{note}\n\n**Steps**\n\n{steps}"))

    rows = []
    for x, y in zip(a["ingredients"], b["ingredients"]):
        slot = x["position"]
        rows.append({
            "ingredient": names[slot],
            "raw qty": quantities[slot] if slot < len(quantities) else "",
            "thinking": fmt(x),
            "no thinking": fmt(y),
            "": "≠" if not same(x, y) else "",
        })
    table = pd.DataFrame(rows)
    display(table.style.hide(axis="index").apply(
        lambda r: ["background-color: #fff3bf; color: #222" if r[""] else "" for _ in r], axis=1))

    if len(quantities) > len(names):
        display(Markdown(f"Raw quantities with no ingredient name: `{quantities[len(names):]}`"))
    pairs = pair_extras(a["extra_ingredients"], b["extra_ingredients"])
    th = ", ".join(f"{n} ({fmt(x)})" for n, x, y in pairs if x) or "none"
    nt = ", ".join(f"{n} ({fmt(y)})" for n, x, y in pairs if y) or "none"
    flag = lambda ok: "" if ok else "≠ "
    display(Markdown(
        f"{flag(all(x and y and same(x, y) for _, x, y in pairs))}**Extras added** — thinking: {th} · no thinking: {nt}  \n"
        f"{flag(a['cuisines'] == b['cuisines'])}**Cuisines** — thinking: {', '.join(a['cuisines'])} · no thinking: {', '.join(b['cuisines'])}"))

In [4]:
card(1, order[0])

### 1. Lemon Cucumbers  
equal · 8 ingredients, 8 raw quantities

**Steps**

1. Place cucumbers and onion in a bowl.
2. Combine rest of the ingredients in a measuring cup.
3. Stir well.
4. Pour over cucumbers and onions.
5. Chill for 4 hours.

ingredient,raw qty,thinking,no thinking,
cucumbers,2,2 piece,2 piece,
yellow onion,1⁄2,0.5 piece,0.5 piece,
vinegar,1⁄4,0.25 cup,0.25 cup,
sugar,2,2 tbsp,2 tbsp,
bottled lemon juice,1,1 tbsp,1 cup,≠
celery seed,1,1 tsp,1 tsp,
salt,3⁄4,0.75 tsp,0.75 tsp,
black pepper,1⁄8,0.125 tsp,0.125 tsp,


**Extras added** — thinking: none · no thinking: none  
≠ **Cuisines** — thinking: other · no thinking: american

In [5]:
verdicts[1] = ""  # thinking | nothinking | both | neither
notes[1] = ""

In [6]:
card(2, order[1])

### 2. Sour Pickle and Sweet Peppers Coleslaw  
equal · 7 ingredients, 7 raw quantities

**Steps**

1. In large bowl, combine all ingredients very well. Season to taste with salt and pepper. Keep refrigerated.

ingredient,raw qty,thinking,no thinking,
red cabbage,1⁄2,0.5 piece,0.5 piece,
red bell peppers,2,2 piece,2 piece,
onion,1⁄2,0.5 piece,0.5 piece,
dill pickle,1,1 piece,1 piece,
fat-free mayonnaise,1,1 cup,1 cup,
apple cider vinegar,1,1 tbsp,1 tbsp,
celery seed,1⁄2,0.5 tsp,0.5 tsp,


**Extras added** — thinking: pepper (to_taste) · no thinking: pepper (to_taste)  
**Cuisines** — thinking: american · no thinking: american

In [7]:
verdicts[2] = ""  # thinking | nothinking | both | neither
notes[2] = ""

In [8]:
card(3, order[2])

### 3. Mashed Potatoes With Sour Cream  
equal · 5 ingredients, 5 raw quantities

**Steps**

1. Place potatoes in a heavy saucepan.
2. Cover with salted water.
3. Bring to a boil over medium high heat. Reduce heat to medium low.
4. Cover saucepan and simmer 15 minutes or until potatoes are tender. Drain, reserving 1/3 cup liquid.
5. Add butter, sour cream, and salt and pepper to taste.
6. Mash until smooth with an electric mixer or hand masher, adding reserved cooking liquid as necessary to moisten potatoes.
7. Stir in parsley. Serve potatoes with mushroom gravy.

ingredient,raw qty,thinking,no thinking,
russet potato,1,1 piece,1 piece,
unsalted butter,2,2 tbsp,2 tbsp,
unsalted butter,2,2 tbsp,2 tbsp,
sour cream,1⁄3,0.33 cup,0.3333 cup,
parsley,2,2 tbsp,2 tbsp,


**Extras added** — thinking: none · no thinking: none  
**Cuisines** — thinking: american · no thinking: american

In [9]:
verdicts[3] = ""  # thinking | nothinking | both | neither
notes[3] = ""

In [10]:
card(4, order[3])

### 4. Mama's Lasagna  
equal · 8 ingredients, 8 raw quantities

**Steps**

1. Brown ground beef in a frying pan.
2. Add sauce and tomato paste.
3. In a bowl mix cottage cheese with egg, Parmesan cheese, and parsley.
4. Cover the bottom of a 9x13 inch pan with a little of the sauce.
5. Layer the uncooked noodles, then cottage cheese mixture, then meat sauce, then slices of mozzarella. Do one more layer in the same order.
6. Sprinkle top with Parmesan cheese.
7. Cover with foil and bake at 350°F for 45 minutes to an hour to make sure noodles are done.

ingredient,raw qty,thinking,no thinking,
ground beef,1,1 lb,1 lb,
tomato sauce,1,1 can,1 can,
tomato paste,2,2 can,2 can,
cottage cheese,1,1 package,1 lb,≠
ricotta cheese,1,1 package,1 lb,≠
egg,1⁄2,0.5 piece,1 piece,≠
fresh parsley,,to_taste,0.5 bunch,≠
mozzarella cheese,1,1 package,1 lb,≠


≠ **Extras added** — thinking: lasagna noodles (1 package), parmesan cheese (to_taste) · no thinking: lasagna noodles (to_taste), parmesan cheese (to_taste)  
**Cuisines** — thinking: italian · no thinking: italian

In [11]:
verdicts[4] = ""  # thinking | nothinking | both | neither
notes[4] = ""

In [12]:
card(5, order[4])

### 5. Mom's Classic Tuna Melt  
equal · 4 ingredients, 4 raw quantities

**Steps**

1. Preheat your oven on low broil.
2. Drain the water out of the tuna and combine the tuna and mayo in a bowl.
3. Cut the bagel in half to form two donuts and toast in a toaster.
4. Spread tuna on each half of the bagel and top with a slice of Swiss cheese.
5. Place in the oven until the cheese is melted.

ingredient,raw qty,thinking,no thinking,
tuna in water,1,1 can,1 can,
mayonnaise,3,3 tbsp,3 tbsp,
swiss cheese,2,2 slice,2 slice,
bagel,1,1 piece,1 piece,


**Extras added** — thinking: none · no thinking: none  
**Cuisines** — thinking: american · no thinking: american

In [13]:
verdicts[5] = ""  # thinking | nothinking | both | neither
notes[5] = ""

In [14]:
card(6, order[5])

### 6. Fava Beans Just Like Grandma Made  
equal · 8 ingredients, 8 raw quantities

**Steps**

1. Shell the beans, add to boiling water, cook for apprx 10 minutes, dash into cold water drain.
2. Heat the oil and butter in a skillet.  Add onion and garlic and saute over medium heat until the onions are golden brown apprx 20 minutes.
3. Add Fava beans, tomato sauce, and pepper to the onions.
4. Place beans and onions in an oven proof dish and sprinkle with cheese.
5. To  make this dish ahead stop here and refrigerate.
6. Bake in 350 F oven for 20 minutes if beans are hot off the stove and 30 minutes if they have been in the frig.
7. Cheese will be melted, if you would like it browned will need to place under the broiled for a minute or two.

ingredient,raw qty,thinking,no thinking,
fava beans,2,2 lb,2 lb,
onion,1,1 piece,1 piece,
butter,1,1 tbsp,1 tbsp,
olive oil,1,1 tbsp,1 tbsp,
garlic clove,1,1 clove,1 clove,
tomato sauce,8,8 oz,8 oz,
pepper,1⁄8,0.125 tsp,0.125 tsp,
parmesan cheese,1,1 cup,1 tbsp,≠


**Extras added** — thinking: none · no thinking: none  
**Cuisines** — thinking: italian · no thinking: italian

In [15]:
verdicts[6] = ""  # thinking | nothinking | both | neither
notes[6] = ""

In [16]:
card(7, order[6])

### 7. Spicy Carrot Coins  
equal · 10 ingredients, 10 raw quantities

**Steps**

1. In nonstick saucepan combine the first 8 ingredients. Bring to a boil. Reduce heat; cover and simmer for 5 minutes.
2. Discard cinnamon stick.
3. Add honey and lemon juice to carrots. Bring to a boil; cook, uncovered, for 5 to 8 minutes or until carrots are tender and liquid has evaporated.

ingredient,raw qty,thinking,no thinking,
carrots,2,2 lb,2 lb,
water,1 1⁄4,1.25 cup,1.25 cup,
cinnamon stick,1,1 piece,1 piece,
ground ginger,1,1 tsp,1 tsp,
ground cumin,1,1 tsp,1 tsp,
ground coriander,1⁄2,0.5 tsp,0.5 tsp,
salt,1⁄4,0.25 tsp,0.25 tsp,
cayenne pepper,1⁄8,0.125 tsp,0.125 tsp,
honey,1⁄4,0.25 cup,0.25 cup,
lemon juice,4,4 tsp,4 tbsp,≠


**Extras added** — thinking: none · no thinking: none  
≠ **Cuisines** — thinking: middle_eastern · no thinking: american

In [17]:
verdicts[7] = ""  # thinking | nothinking | both | neither
notes[7] = ""

In [18]:
card(8, order[7])

### 8. Black Bean Chile  
more · 9 ingredients, 10 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Saute onion and garlic in olive oil.
2. Add jalapeno pepper, if desired; this chile is spicy enough for most people without the jalapeno.
3. Combine all ingredients, except the cilantro, in a pot or crockpot.
4. Cook on low heat until the chile is warmed through.
5. Add cilantro about 15 minutes before serving.
6. Leftovers can be frozen and reheated later.

ingredient,raw qty,thinking,no thinking,
onion,1,1 piece,1 piece,
garlic,2,2 clove,2 clove,
olive oil,1,2 tbsp,2 tbsp,
fresh cilantro,1,1 bunch,0.25 bunch,≠
tomatoes and green chilies,1,1 can,1 can,
black beans,5,2 can,1 can,≠
diced green chilies,2,1 can,1 can,
oregano,1,1 tsp,1 tsp,
cumin,1,1 tsp,1 tsp,


Raw quantities with no ingredient name: `['1']`

≠ **Extras added** — thinking: jalapeno pepper (1 piece) · no thinking: jalapeno pepper (to_taste)  
≠ **Cuisines** — thinking: tex_mex · no thinking: mexican, tex_mex

In [19]:
verdicts[8] = ""  # thinking | nothinking | both | neither
notes[8] = ""

In [20]:
card(9, order[8])

### 9. Mussels in Spicy Coconut Broth  
more · 8 ingredients, 9 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Combine first 7 ingredients in a Dutch oven, and bring to a boil. Add mussels; cover and cook 5 minutes or until shells open.
2. Remove from heat; discard any unopened shells. Spoon 1 1/2 cups broth mixture into each of 2 soup bowls.
3. Divide mussels evenly among each soup bowl. Garnish with chopped basil.

ingredient,raw qty,thinking,no thinking,
light coconut milk,1⁄2,1 can,1.5 cup,≠
fresh ginger,1⁄4,1 tbsp,1 piece,≠
sugar,1,1 tsp,0.5 tsp,≠
lemon juice,2,1 tbsp,0.5 tbsp,≠
curry powder,2,0.5 tsp,0.5 tsp,
salt,1,to_taste,to_taste,
mussels,1,1 lb,1 lb,
fresh basil,2,2 tbsp,2 tbsp,


Raw quantities with no ingredient name: `['1⁄4']`

**Extras added** — thinking: none · no thinking: none  
≠ **Cuisines** — thinking: thai · no thinking: thai, indian

In [21]:
verdicts[9] = ""  # thinking | nothinking | both | neither
notes[9] = ""

In [22]:
card(10, order[9])

### 10. Cherokee Casserole  
more · 11 ingredients, 12 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Cook rice per normal cooking instructions.
2. While rice cooks, brown meat and cook onions until tender.
3. Drain excess fat from meat.
4. Combine meat and onions in skillet, and add crushed tomatoes, mushroom soup, bay leaf, oregano, garlic powder, cumin, salt, pepper, olives.
5. Simmer together in skillet until rice is finished cooking.
6. Add rice, and mix all ingredients well.
7. Top casserole with cheese until melted.

ingredient,raw qty,thinking,no thinking,
long-grain rice,1,1 cup,1 cup,
tomatoes,1,1 can,1 can,
ground beef,1,1 lb,1 lb,
onion,1,1 piece,1 piece,
bay leaf,1,1 piece,1 piece,
oregano,1⁄2,0.5 tsp,0.5 tsp,
garlic powder,1⁄8,0.125 tsp,0.125 tsp,
cumin,1⁄8,0.125 tsp,0.125 tsp,
green olives,1⁄8,0.125 cup,0.125 cup,
cheddar cheese,12,12 oz,12 oz,


Raw quantities with no ingredient name: `['']`

**Extras added** — thinking: mushroom soup (1 can), pepper (to_taste) · no thinking: mushroom soup (1 can), pepper (to_taste)  
≠ **Cuisines** — thinking: native_american, american · no thinking: american, tex_mex

In [23]:
verdicts[10] = ""  # thinking | nothinking | both | neither
notes[10] = ""

In [24]:
card(11, order[10])

### 11. Acorn Squash, Pear and Azuki Soup With Sauteed Shiitakes  
more · 10 ingredients, 16 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Preheat a large stock pot over medium heat. Add the oil and sauté the onions and peppers for about 10 minutes, or until onions just begin to brown.
2. Add the ginger and garlic and sauté a minute more. Add the salt, Chinese five spice, acorn squash and pear and cook for another minute, stirring often. Add the vegetable broth. Cover and bring to a boil. Once boiling, lower the heat just a bit to simmer briskly for about 20 more minutes, or until the squash is tender.
3. Puree half the soup using either an immersion blender or by transferring half the soup to a food processor or blender and then back into the rest of the soup (don’t forget to let the soup cool a bit so that the steam does not compress in the processor and kill you).
4. Add the azuki beans and lime. Cover and simmer on low, just until the beans are heated through. Meanwhile prepare the mushrooms.
5. Preheat a heavy bottomed skillet over medium high heat. Add the oils and sauté the mushrooms for about 7 minutes, until they are soft. Add the soy sauce and stir constantly until the soy sauce is absorbed (about 1 minute).
6. Ladle soup into bowls and top with sautéed mushrooms.

ingredient,raw qty,thinking,no thinking,
yellow onion,2,1 piece,1 piece,
garlic cloves,1,3 clove,2 clove,≠
salt,1,to_taste,to_taste,
Chinese five spice powder,2,0.5 tsp,0.5 tsp,
acorn squash,2,1 piece,1 piece,
bartlett pears,1⁄2,2 piece,2 piece,
vegetable broth,1⁄2,1 l,4 cup,≠
azuki beans,2,1 cup,1 can,≠
fresh lime juice,2,2 tbsp,2 tbsp,
soy sauce,4,2 tbsp,2 tbsp,


Raw quantities with no ingredient name: `['15', '1', '4', '2', '1⁄2', '1']`

≠ **Extras added** — thinking: oil / cooking oil (2 tbsp), fresh ginger / ginger (1 tbsp), shiitake mushrooms (250 g) · no thinking: oil / cooking oil (2 tbsp), fresh ginger / ginger (1 tbsp), shiitake mushrooms (8 oz)  
≠ **Cuisines** — thinking: chinese · no thinking: chinese, other

In [25]:
verdicts[11] = ""  # thinking | nothinking | both | neither
notes[11] = ""

In [26]:
card(12, order[11])

### 12. Linguine With Sun Dried Tomato Sauce  
more · 6 ingredients, 8 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. In large skillet, heat oil from tomatoes. Add garlic and cook over medium heat. Add canned tomatoes and sun-dried tomatoes and bring to a boil over medium heat.
2. Reduce heat and simmer, 5 minutes. Add cream and cook 5 minutes to blend flavors.
3. Meanwhile cook pasta. Drain.
4. Season sauce with pepper and stir in vinegar. Spoon over pasta and sprinkle with cheese.

ingredient,raw qty,thinking,no thinking,
sun-dried tomato packed in oil,3⁄4,0.75 cup,0.75 cup,
garlic cloves,3,3 clove,3 clove,
crushed tomatoes in puree,28,28 oz,28 oz,
heavy cream,1⁄3,0.33 cup,0.333 cup,
pepper,9,to_taste,to_taste,
parmesan cheese,1⁄4,0.25 cup,0.25 cup,


Raw quantities with no ingredient name: `['1', '1⁄4']`

≠ **Extras added** — thinking: vinegar / balsamic vinegar (to_taste), linguine (1 lb) · no thinking: vinegar / balsamic vinegar (1 tbsp), linguine (9 oz), olive oil (to_taste)  
**Cuisines** — thinking: italian · no thinking: italian

In [27]:
verdicts[12] = ""  # thinking | nothinking | both | neither
notes[12] = ""

In [28]:
card(13, order[12])

### 13. Cheesy Scalloped Corn Bake  
more · 11 ingredients, 13 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Set oven to 350 degrees.
2. Butter a 1-quart casserole dish.
3. In a bowl mix all ingredients together until well blended.
4. Transfer to prepared casserole dish.
5. Bake uncovered for 30-35 minutes.
6. Remove and top with shredded cheddar cheese (any amount desired) return to oven for 3-5 minutes or until cheese is melted.

ingredient,raw qty,thinking,no thinking,
eggs,1,2 piece,2 piece,
butter,2,2 tbsp,2 tbsp,
margarine,1⁄2,0.25 cup,2 tbsp,≠
evaporated milk,1⁄4,1 can,0.5 can,≠
green bell pepper,1⁄4,0.5 piece,0.25 piece,≠
celery,1,0.25 cup,0.25 piece,≠
onion,1,0.25 piece,1 piece,≠
sugar,1,1 tbsp,1 tsp,≠
seasoning salt,1,to_taste,1 tsp,≠
pepper,3,to_taste,3 pinch,≠


Raw quantities with no ingredient name: `['', '1']`

≠ **Extras added** — thinking: none · no thinking: cheddar cheese, shredded (for topping) (to_taste)  
**Cuisines** — thinking: american · no thinking: american

In [29]:
verdicts[13] = ""  # thinking | nothinking | both | neither
notes[13] = ""

In [30]:
card(14, order[13])

### 14. Barley Risotto With Caramelized Leeks and Mushrooms  
more · 7 ingredients, 8 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Heat the oil in a large saucepan over medium heat. Add leek; cook 20 minutes or until tender and golden, stirring occasionally.
2. Add mushrooms; cook 5 minutes or until tender, stirring occasionally. Add barley, salt, and pepper; cook 1 minute, stirring constantly.
3. Add broth; bring to a boil. Cover, reduce heat, and simmer 30 minutes, stirring occasionally.
4. Uncover and cook 5 minutes or until liquid is absorbed. Stir in parsley.

ingredient,raw qty,thinking,no thinking,
olive oil,2,2 tbsp,2 tbsp,
leeks,3,3 piece,3 piece,
cremini mushrooms,3,8 oz,3 cup,≠
pearl barley,1,1.5 cup,1 cup,≠
salt,1,to_taste,to_taste,
fresh ground black pepper,1⁄4,to_taste,to_taste,
fresh flat-leaf parsley,2,1 bunch,0.25 cup,≠


Raw quantities with no ingredient name: `['2']`

≠ **Extras added** — thinking: broth / vegetable broth (4 cup) · no thinking: broth / vegetable broth (2 l)  
**Cuisines** — thinking: italian · no thinking: italian

In [31]:
verdicts[14] = ""  # thinking | nothinking | both | neither
notes[14] = ""

In [32]:
card(15, order[14])

### 15. Better Tater Tot Casserole  
more · 7 ingredients, 10 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Brown ground beef.
2. Add onions and garlic.
3. Fry until onions are translucent.
4. Add green beans, both soups, milk, and tater tots, reserving about 2 cups of tater tots for the top.
5. mix well.
6. place mixture in casserole dish.
7. place reserved tater tots on top, pushing in slightly.
8. sprinkle seasoning salt on top.
9. sprinkle grated cheese on top.
10. bake uncovored at 350 for 30 minutes.

ingredient,raw qty,thinking,no thinking,
green beans,2,2 can,2 cup,≠
onion,1,1 piece,1 piece,
garlic cloves,1,2 clove,2 clove,
seasoning salt,2,to_taste,to_taste,
ground beef,1⁄2,1 lb,1 lb,
cheddar cheese,2,2 cup,2 cup,
milk,1,1 cup,1 cup,


Raw quantities with no ingredient name: `['1', '1⁄2', '1⁄2']`

≠ **Extras added** — thinking: condensed soup (2 can), tater tots (4 cup) · no thinking: tater tots (to_taste), cream of mushroom soup (1 can), cream of chicken soup (1 can)  
**Cuisines** — thinking: american · no thinking: american

In [33]:
verdicts[15] = ""  # thinking | nothinking | both | neither
notes[15] = ""

In [34]:
card(16, order[15])

### 16. Black Pepper Pork with Ramen Noodles  
more · 5 ingredients, 10 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Cut pork across grain into 1/4 inch wide strips; coat with mixture of garlic and 1 tbs. Stir-Fry Sauce.
2. Let stand 15 minutes.
3. Meanwhile, cook noodles only (not seasoning packet) in 2 qts boiling water 2 minutes; drain.
4. Heat 1 tbs oil in hot wok or large skillet over high heat.
5. Add pork and stir-fry 2 minutes; remove.
6. Reduce heat to medium-high; heat remaining vegetable oil in same pan.
7. Add onions; stir fry 1 minute.
8. Add broccoli and 2 tbs water; stir-fry 3 minutes.
9. Stir in pork, noodles, and remaining stir-fry sauce and pepper.
10. Cook, stirring well, until all ingredients are coated with sauce.
11. Add sesame oil and toss well to combine.

ingredient,raw qty,thinking,no thinking,
boneless pork chop,3⁄4,1 lb,0.75 lb,≠
garlic,2,2 clove,2 clove,
onion,1⁄3,1 piece,0.333 piece,≠
broccoli florets,1,2 cup,1 cup,≠
black pepper,2,to_taste,to_taste,


Raw quantities with no ingredient name: `['2', '1', '4', '1⁄2', '1']`

≠ **Extras added** — thinking: ramen noodles (1 package), vegetable oil (2 tbsp), stir-fry sauce (2 tbsp), sesame oil (to_taste) · no thinking: ramen noodles (1 package), vegetable oil (2 tbsp), stir-fry sauce (2 tbsp), sesame oil (1 tsp)  
**Cuisines** — thinking: chinese · no thinking: chinese

In [35]:
verdicts[16] = ""  # thinking | nothinking | both | neither
notes[16] = ""

In [36]:
card(17, order[16])

### 17. Stir Fry Vegan Ramen  
more · 10 ingredients, 11 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. cooked instant noodles as package drain and set aside.
2. with sesame oil brown onion then add carrot saute for one minute then noodle.
3. add sauce and half cup of water and mix well then add beansprouts with spring onion for 2 minute.
4. if its too dry add all water then mix well til almost dry.
5. off heat top black pepper and mix well and serve.

ingredient,raw qty,thinking,no thinking,
water,3,0.5 cup,0.5 cup,
black pepper,1,to_taste,to_taste,
bean sprouts,1,100 g,100 g,
carrot,1,1 piece,1 piece,
spring onion,100,1 bunch,2 piece,≠
chili peppers,1,1 piece,1 piece,
light soy sauce,1,3 tbsp,3 tbsp,
peri-peri,3,1 tbsp,1 tbsp,
chili sauce,3,3 tbsp,3 tbsp,
dark soy sauce,2,2 tbsp,2 tbsp,


Raw quantities with no ingredient name: `['1']`

≠ **Extras added** — thinking: sesame oil (1 tbsp), brown onion / onion (1 piece), instant noodles (1 package) · no thinking: sesame oil (to_taste), brown onion / onion (1 piece), instant noodles (1 package)  
≠ **Cuisines** — thinking: japanese · no thinking: chinese, other

In [37]:
verdicts[17] = ""  # thinking | nothinking | both | neither
notes[17] = ""

In [38]:
card(18, order[17])

### 18. Smoked Salmon and Basil Rolls with Creme Fraiche or Sour Cream  
fewer · 5 ingredients, 4 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Place 1 salmon slice on work surface.
2. Spread 2 teaspoons creme fraiche over salmon.
3. Sprinkle with 2 teaspoons chopped basil.
4. sprinkle with pepper.
5. Starting at 1 short end, roll up salmon slice, enclosing filling.
6. Spread 1 teaspoon creme fraiche over top of roll; sprinkle with 1 teaspoon chopped basil.
7. Cut roll crosswise into 5 slices (keep sliced roll together).
8. Repeat with remaining salmon, creme fraiche, and chopped basil.
9. (Can be made 6 hours ahead. Transfer salmon rolls to plate, cover and chill.) Arrange whole basil leaves on platter.
10. Using small spatula, top each basil leaf with 1 sliced salmon roll, fanning slightly.

ingredient,raw qty,thinking,no thinking,
smoked salmon,12,12 piece,12 piece,
creme fraiche,3⁄4,36 tsp,0.75 cup,≠
sour cream,3⁄4,to_taste,0.75 cup,≠
fresh basil,12,36 tsp,12 piece,≠
fresh basil leaves,,12 piece,to_taste,≠


**Extras added** — thinking: black pepper / pepper (to_taste) · no thinking: black pepper / pepper (to_taste)  
≠ **Cuisines** — thinking: other · no thinking: american

In [39]:
verdicts[18] = ""  # thinking | nothinking | both | neither
notes[18] = ""

In [40]:
card(19, order[18])

### 19. Special Thai Chicken Salad  
fewer · 20 ingredients, 18 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Mix the marinade ingredients together in a cup except for the ground pepper. Slather over chicken, ensuring all parts are equally covered with marinade. Then press coursely-ground peppercorn all over the chicken's surface. Allow to marinate in the refrigerator for at least 10 to 15 minutes (or up to 24 hours).
2. If Using the Oven: Set oven to BROIL and place chicken on a broiling pan or a foil-lined baking sheet (turn out the sides of the foil to prevent juices from spilling into oven).
3. Place chicken on second-to-highest rung of your oven, allowing it to cook 5 minutes each side, then turning. Continue broiling and turning until cooked (chicken will be lightly charred around the edges). Use the leftover marinade to baste chicken the first time you turn it.
4. If Grilling the Chicken: Place chicken over a hot grill and cook until well done. Use leftover marinade to baste chicken when you turn it the first time.
5. While chicken is cooking, prepare the salad greens and dressing. To make the dressing: place water, lemongrass and lime leaf strips (or zest) in a sauce pan and bring to a boil. Boil for 1 minute (this will soften the lemongrass and bring out the flavor of the lime leaf). Remove from heat.
6. Add the rest of the dressing ingredients and mix well.
7. Taste-test the dressing. It should be tangy - a balance of salty, sweet, spicy, and sour. Adjust it to your taste by adding more fish sauce if not salty enough, more lime juice if too salty or sweet, more sugar if too sour or more chili sauce if not spicy enough for your taste.
8. Toss the salad with the dressing. Portion out the salad among individual plates, and top with strips of the hot chicken. Sprinkle with fresh basil and coriander; serve immediately while chicken is still warm.

ingredient,raw qty,thinking,no thinking,
boneless chicken breasts,2 -4,3 piece,3 piece,
chicken thighs,,2 piece,2 piece,
fresh basil,1⁄2,1 bunch,0.5 cup,≠
fresh cilantro,4,1 bunch,0.5 cup,≠
fish sauce,6,4 tbsp,2 tbsp,≠
garlic cloves,1,6 clove,4 clove,≠
soy sauce,2,2 tbsp,2 tbsp,
lemons,1,1 piece,1 piece,
lime juice,1⁄4,0.25 cup,2 tbsp,≠
fresh ground black pepper,1,to_taste,to_taste,


**Extras added** — thinking: none · no thinking: none  
**Cuisines** — thinking: thai · no thinking: thai

In [41]:
verdicts[19] = ""  # thinking | nothinking | both | neither
notes[19] = ""

In [42]:
card(20, order[19])

### 20. Avocado Lime Rice With Grilled Shrimp  
fewer · 11 ingredients, 10 raw quantities  
_Raw quantities may be shifted for this recipe, so lean on the steps._

**Steps**

1. Cook rice according to package directions. Meanwhile mix avocado with lime juice, cumin and half the salt and half the cilantro or parsley. Stir into rice.
2. Meanwhile, preheat grill to high, grease grate well. Toss shrimp with oil, garlic, lime zest and remaining salt. Thread evenly onto metal or soaked wooden skewers. Grill skewers, for 2 to 3 minutes per side or until shrimp start to curl and turn pink.
3. Serve rice with shrimp skewers. Garnish with remaining cilantro or parsley.

ingredient,raw qty,thinking,no thinking,
long grain white rice,2,1.5 cup,2 cup,≠
avocado,1,1 piece,1 piece,
lime juice,2,2 tbsp,2 tbsp,
ground cumin,1,1 tsp,1 tsp,
salt,1⁄2,to_taste,0.5 tsp,≠
fresh cilantro,2,1 bunch,0.25 bunch,≠
parsley,20,0.5 bunch,0.25 bunch,≠
frozen shrimp,2,20 piece,20 piece,
olive oil,1,2 tbsp,2 tbsp,
garlic,2,2 clove,2 clove,


**Extras added** — thinking: none · no thinking: none  
≠ **Cuisines** — thinking: tex_mex · no thinking: mexican, american

In [43]:
verdicts[20] = ""  # thinking | nothinking | both | neither
notes[20] = ""

## Your verdicts

In [44]:
review = pd.DataFrame([
    {
        "n": i,
        "recipe": stage1[key]["name"],
        "group": stage1[key]["group"],
        "ingredient entries that differ": f"{n_differ(key)} of {len(stage1[key]['ingredient_parts'])}",
        "verdict": verdicts.get(i, ""),
        "note": notes.get(i, ""),
    }
    for i, key in enumerate(order, 1)
])
review.to_csv(HERE / "review.csv", index=False)
filled = review[review["verdict"] != ""]
print(f"saved review.csv ({len(filled)} of 20 verdicts filled)")
if len(filled):
    display(filled["verdict"].value_counts().rename("recipes").to_frame())
    display(pd.crosstab(filled["group"], filled["verdict"]))

saved review.csv (0 of 20 verdicts filled)
